# 🧠 NeuroPET detective: line up the brain images

**A 45–50 minute, click-and-run investigation**

Imagine that we have taken two different pictures of the same brain. They contain different clues, and we want to combine them.

**MRI** uses a strong magnetic field and radio waves to make detailed pictures of anatomy. It is very good for seeing the brain's shape, ventricles, and boundaries between tissues.

**PET** uses a small amount of radioactive tracer. A PET camera maps where that tracer collects. PET looks blurrier than MRI, but it can tell us important things about what the brain is doing—for example, how it uses glucose or where particular molecules have accumulated.

The two scans may be acquired on different days and in different scanners. What if the person's head is turned or shifted slightly between them? A bright PET location would no longer sit over the correct anatomy. **Registration** is the process of moving images into alignment before we measure or interpret them.

Today you are the image-registration team. You will align MRI and PET, ask a computer to check your work, and measure tracer signal.

> The example images were created from a digital brain model for this activity; they are not scans of a real person.

<div style="padding:12px 16px;border-left:6px solid #31688e;background:#eef6fb"><b>The mission</b><br>MRI + PET → registration → quality check → segmentation → measurement → interpretation</div>

## 1. RUN THIS — open the imaging toolkit

Click the cell below, then press **Shift + Enter**. The code is complete; you do not need to type anything.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from workshop_helpers import (
    animate_rough_optimiser, compare_cases, how_well_matched, load_workshop_data,
    normalise_match_score,
    measure_gm_uptake, optimise, shift_image,
    show_alignment, show_measurement, show_score,
)

data = load_workshop_data()
mri = data["mri"]
gm_mask = data["gm_mask"]
gm_contour = data["gm_probability"]
low_binding_pet = data["low_binding_pet"]
high_binding_pet = data["high_binding_pet"]
challenge_pet = data["challenge_pet"]
print("✓ Toolkit ready.")

## 2. Meet the images

- **MRI:** shows anatomical structure clearly.
- **PET:** shows where the tracer has collected. It is blurrier, but it gives different information.

### PREDICT

Which image will show the ventricles and tissue boundaries most clearly? Which will show tracer distribution?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
axes[0].imshow(mri, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("MRI: detailed anatomy", fontsize=14)
pet_image = axes[1].imshow(low_binding_pet, cmap="magma", vmin=0, vmax=1.2)
axes[1].set_title("PET: tracer distribution", fontsize=14)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(pet_image, ax=axes[1], shrink=0.82, label="Relative tracer signal")
fig.suptitle("Two images, two kinds of information", fontsize=16, fontweight="bold");

<details><summary><b>CHECK YOUR ANSWER</b></summary><br>The MRI shows anatomy most clearly. The PET shows tracer distribution and is visibly blurrier.</details>

## 3. Manual registration challenge

MRI and PET are often acquired separately—sometimes on different days and in different scanners. Even when the same person is scanned, their head will not be in exactly the same position each time. The images therefore may not line up when we first open them.

That has happened here: the PET image has been shifted away from the MRI. Your job is to register it by moving it back into alignment. The cyan contour is a segmentation marking grey-matter tissue in the MRI; it is not a named brain region.

**Sign convention:** positive `x` moves the image right; positive `y` moves it down. Try values from **−10 to +10**.

### CHANGE ONLY THE TWO NUMBERS BELOW

Start at zero, inspect the fused image, then change the two numbers and run the cell again. Aim to place the PET edge and internal features inside the cyan contour.

In [ ]:
x_offset = 0
y_offset = 0

student_pet = shift_image(challenge_pet, x_offset, y_offset)
show_alignment(mri, student_pet, gm_contour, title="Your manual registration");

## 4. Quantify your match

Eyes are useful, but a computer needs a number. We calculate normalized mutual information (NMI), which asks how strongly the pattern in one image is related to the pattern in the other—even when MRI and PET use different brightness scales. We then rescale the useful NMI range for this exercise so the displayed **match score** runs from 0 to 1. Larger is better; it is not percentage accuracy.

Run the cell, then return to the two-number cell and try to improve your score two or three times.

In [ ]:
student_score = how_well_matched(mri, student_pet)
show_score(student_score, label="Your match score");

<details><summary><b>NEED A HINT?</b></summary><br>Look at the brain's outer edge first. Decide left/right before up/down. Change one number at a time. A correction close to 7 pixels in each direction should work.</details>

<details><summary><b>SHOW THE MANUAL SOLUTION</b></summary><br>The hidden displacement was 7 pixels right and 5 pixels up, so the exact correction is <code>x_offset = -7</code> and <code>y_offset = 5</code>.</details>

## 5. Let the computer search

First, watch a deliberately rough optimiser. It makes some wild guesses, then takes smaller jumps as it settles. The upper graph tracks its x and y corrections for each guess. The lower graph shows whether its match score is improving.

### PREDICT

Will the computer beat your current score?

In [ ]:
rough_animation = animate_rough_optimiser(mri, challenge_pet, gm_contour)
display(rough_animation)

The animation is intentionally scrappy so we can see the search. For the final answer below, our reliable optimiser tests every whole-pixel correction from −10 to +10: **21 × 21 = 441 possible shifts**. It keeps the shift with the largest match score.

In [ ]:
automatic_pet, best_x, best_y, automatic_score = optimise(mri, challenge_pet, search_range=10)
student_display_score = normalise_match_score(student_score)
automatic_display_score = normalise_match_score(automatic_score)
print(f"Your correction:      x={x_offset:+d}, y={y_offset:+d} | score {student_display_score:.3f} / 1")
print(f"Computer correction:  x={best_x:+d}, y={best_y:+d} | score {automatic_display_score:.3f} / 1")
print("The optimiser tested 441 possible shifts.")
show_alignment(mri, automatic_pet, gm_contour, title="Automatic registration result");

### WHAT DID YOU NOTICE?

- Did the optimiser beat or equal your result?
- What extra possibilities would it need to test if the PET could also rotate or change size?

<details><summary><b>CHECK YOUR ANSWER</b></summary><br>The computer should find x = −7 and y = +5. Allowing rotation or scaling would create many more candidate transformations, so the search would take longer.</details>

## 6. Measure uptake

Now the images are aligned, use the grey-matter mask to choose measurement pixels. The number is their **mean relative tracer signal**. It has no clinical units.

In [ ]:
registered_uptake = measure_gm_uptake(automatic_pet, gm_mask)
poorly_registered_uptake = measure_gm_uptake(challenge_pet, gm_mask)
print(f"After registration:  {registered_uptake:.3f}")
print(f"Before registration: {poorly_registered_uptake:.3f}")
print(f"Difference caused by misalignment: {registered_uptake - poorly_registered_uptake:+.3f}")
show_measurement(automatic_pet, gm_mask, title="Where did the measurement come from?");

<details><summary><b>WHY DOES REGISTRATION MATTER?</b></summary><br>Without registration, the fixed grey-matter mask samples some wrong locations and misses some intended ones. A plausible-looking image can therefore produce a biased number.</details>

## 7. Compare two examples

Both examples use the same anatomy, scale, blur, and noise pattern. Only the grey-matter binding level differs. Keeping the colour limits identical makes the comparison fair.

### PREDICT

Which example will have the larger mean grey-matter signal?

In [ ]:
comparison_figure, low_uptake, high_uptake = compare_cases(
    low_binding_pet, high_binding_pet, gm_mask
)
display(Markdown(
    "| Example | Mean grey-matter uptake | Observation |\n"
    "|---|---:|---|\n"
    f"| Low binding | {low_uptake:.3f} | Less cortical grey-matter signal |\n"
    f"| High binding | {high_uptake:.3f} | More cortical grey-matter signal |"
))

### WHAT DID YOU NOTICE?

1. Which case has greater grey-matter tracer retention?
2. Why must both images be registered before comparison?
3. Does one PET measurement alone diagnose Alzheimer's disease?

<details><summary><b>CHECK YOUR ANSWER</b></summary><br><ol><li>The high-binding case has the larger mean.</li><li>Registration makes the mask sample corresponding anatomy rather than different locations.</li><li>No. In an amyloid-PET context, increased cortical retention can contribute evidence of amyloid pathology. A real diagnosis also needs appropriate patient selection, expert image interpretation, symptoms, history, examination, cognitive assessment, and other clinical information.</li></ol></details>

## 8. Recap

<div style="text-align:center;font-size:1.25em;padding:16px;background:#f3f1fa;border-radius:8px"><b>MRI + PET → registration → quality check → segmentation → measurement → interpretation</b></div>

Explain the workflow back in one sentence. Where could an error at the start change the final interpretation?

<details><summary><b>ONE POSSIBLE SUMMARY</b></summary><br>We align a tracer image to anatomy, check the alignment, use a tissue mask to choose pixels, measure their mean signal, and interpret that result alongside—not instead of—other clinical evidence.</details>

---

**Data note:** These workshop images were created from BrainWeb subject 04 using Casper O. da Costa-Luis' Python `brainweb` package. See `DATA_SOURCES.md` for provenance and licensing details.